# Build a durable, branching AI agent workflow from visible parts

This lab builds a supplier-compliance agent workflow without importing a
preassembled workflow wrapper. The important mechanics are written directly
in cells: Python tools, database registration, semantic retrieval, full
`SKILL.md` storage, sandbox execution, Oracle memory, checkpointing, graph
nodes, routing rules, approval, verification, and workflow-to-skill learning.

The model has a bounded role. It turns verified findings into a clear report.
Deterministic criteria decide which operational path is legal. This division
is useful in regulated work: model reasoning remains valuable, while missing
evidence, blocking sanctions results, and publication authority are enforced
by code rather than suggested in a prompt.

## Industry use case — supplier renewal with three possible trajectories

A manufacturer reviews suppliers before renewing purchasing authority. The
incoming cases are deliberately different:

- **Northstar Textiles** is high risk, represents $4.8 million in annual
  spend, has a 401-day-old restricted-substance certificate, and has an open
  audit action. It requires a full review and named human approval.
- **Bluebird Components** is standard risk, represents $240,000 in spend,
  has current evidence, and has no audit or sanctions issue. It qualifies for
  a shorter path and policy-based publication.
- **Contoso Fibres** is missing a required certificate. The workflow must
  stop and request evidence instead of inventing a finding.

For a full review, three independent checks should not wait on one another.
Freshness, sanctions, and audit work therefore run in parallel. Their results
meet at one risk-scoring barrier. The resulting score and blocking flags then
choose remediation, human approval, automatic publication, rejection, or an
evidence request.

![Supplier-renewal workflow showing evidence routing, parallel review checks,
risk scoring, publication authority, and human approval.](../images/Supplier_Renewal_Workflow_OReilly.png)


## Harness component map

A harness is the operational layer around inference. The table separates
model work from control-flow, persistence, memory, retrieval, isolation, and
authority so each responsibility has an inspectable owner.

| Harness component | Provider / implementation | Responsibility in this workflow |
|---|---|---|
| Model | OpenAI Responses API, with a labelled deterministic fallback | Draft only from verified branch outputs |
| Orchestration | LangGraph `StateGraph` defined below | Parallel fan-out, join barrier, conditional routes, interrupt, and terminal states |
| Working checkpoint | LangGraph Oracle `OracleSaver` | Persist graph state and the next executable node for a thread |
| Agent memory | Oracle Agent Memory 26.6 `OracleAgentMemory` | Record durable operational facts and outcomes |
| Memory substrate | `OracleDBMemoryStore` | Store Agent Memory records in Oracle |
| Toolbox | Typed `NB_AGENT_TOOLS` table plus Python `TOOL_REGISTRY` | Retrieve model-facing JSON contracts by meaning and resolve names to callables |
| Skillbox | Typed `NB_AGENT_SKILLS` table | Store complete SHA-versioned `SKILL.md` procedures and retrieve a small manifest |
| Embeddings | OpenAI `text-embedding-3-small`; labelled feature-hash fallback | Encode tool, skill, and query meaning for cosine search |
| Sandbox | Fresh MemoRizz E2B session per tool; isolated subprocess fallback | Execute portable tool logic with no forwarded environment and verify it on the host |
| Human authority | LangGraph `interrupt()` and `Command(resume=...)` | Bind approve/reject to the exact high-risk report |
| Learning | Typed workflow recipe rows and an explicit promotion rule | Promote only recurring, verified trajectories into a new Skill |

The square below is the database boundary. Checkpoint state, cognitive
memory, and procedural memory are different data products, even though they
share one durable platform.

```mermaid
flowchart TB
  Operator[Compliance operator] --> Graph[LangGraph control flow]
  Model[LLM provider] --> Graph
  Graph --> Registry[Python callable registry]
  Registry --> Sandbox[Fresh tool sandbox]
  Graph --> Approval{Named authority}
  subgraph ORACLE["Oracle AI Database"]
    direction TB
    Checkpoints[OracleSaver checkpoints]
    Memory[OracleDBMemoryStore]
    Skillbox[Skillbox · full SKILL.md + SHA]
    Toolbox[Toolbox · JSON contracts + vectors]
    Recipes[Verified workflow recipes]
  end
  Graph --> Checkpoints
  Graph --> Memory
  Graph --> Skillbox
  Graph --> Toolbox
  Graph --> Recipes
  Recipes --> Promotion[Promotion rule]
  Promotion --> Skillbox
```


## Why a harness rather than only a framework?

LangGraph supplies graph primitives, interrupts, checkpoint integration, and
graph inspection. Oracle Agent Memory supplies memory APIs. Oracle AI
Database supplies transactions, JSON, and vector search. E2B supplies remote
isolation. Those capabilities do not decide the application's operating
rules: which procedures are canonical, which tools enter a model context,
which case requires a full review, what counts as a blocking finding, who may
publish, or when a repeated trajectory has earned promotion.

Those application-specific choices are the harness. Keeping the composition
visible makes the boundary teachable: provider libraries perform specialised
work, graph edges express execution order, and small policy functions express
the organisation's decisions. There is no large orchestration object to read
around. A learner can inspect a node, its inputs, its tool contract, its
sandbox envelope, and its outgoing criteria in the same artifact.

This also makes change safer. Lowering the material-spend threshold changes
one routing function. Adding a new independent check adds one registered tool
and one parallel node. Changing the memory provider does not rewrite risk
policy. Replacing the model does not grant it publication authority.


## Part I — configure Oracle-backed persistence

This notebook has one persistence profile: Oracle AI Database. LangGraph
checkpoints, Agent Memory, tool contracts, Skills, and learned workflow
recipes all use that durable boundary. There is no process-memory
checkpointer or list-backed storage path.

Secrets are read from ignored environment configuration or a hidden prompt.
The notebook reports only whether each secret is configured; it never prints
a credential.


### Import the small set of building blocks

These imports support hashing, JSON contracts, typed graph state, Oracle
vector values, hidden credential prompts, and notebook display. Keeping them
in their own cell makes dependency failures easy to diagnose before any
external connection is opened.


In [1]:
import hashlib
import json
import math
import operator
import os
import re
import subprocess
import sys
import uuid
from array import array
from dataclasses import replace
from getpass import getpass
from importlib import reload
from importlib.metadata import version
from pathlib import Path
from pprint import pprint
from typing import Annotated, Any, TypedDict

from IPython.display import Markdown, display


### Load current project settings

Jupyter keeps imported modules in memory. Reloading the configuration module
ensures this run sees the current `AdvancedSettings` dataclass even if the
project files changed while the kernel remained open. The `replace` call then
makes the notebook's Oracle-only contract explicit.

The three `AGENT_MEMORY_*` values include environment-based defaults for
compatibility with older project configurations. That removes the brittle
direct attribute lookup that previously raised
`AttributeError: 'AdvancedSettings' object has no attribute
'agent_memory_search_strategy'`.


In [2]:
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "part_1" / "advanced").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "part_1" / "advanced").exists():
    raise RuntimeError("Start Jupyter inside the repository root")
sys.path.insert(0, str(REPO_ROOT))

from part_1.advanced.shared import config as advanced_config

advanced_config = reload(advanced_config)
runtime_settings = replace(advanced_config.AdvancedSettings(), backend="oracle")
REMOTE_SANDBOX_MODE = os.environ.get("ADVANCED_SANDBOX_LIVE", "1") == "1"
AGENT_MEMORY_SEARCH_STRATEGY = getattr(
    runtime_settings,
    "agent_memory_search_strategy",
    os.environ.get("ADVANCED_AGENT_MEMORY_SEARCH_STRATEGY", "vector").strip().lower(),
)
AGENT_MEMORY_STORE_ID = getattr(
    runtime_settings,
    "agent_memory_store_id",
    os.environ.get("ADVANCED_AGENT_MEMORY_STORE_ID", "harness_vec1536"),
)
AGENT_MEMORY_EMBED_MAX_TOKENS = int(getattr(
    runtime_settings,
    "agent_memory_embed_max_tokens",
    os.environ.get("ADVANCED_AGENT_MEMORY_EMBED_MAX_TOKENS", "8191"),
))
if AGENT_MEMORY_SEARCH_STRATEGY not in {"keyword", "vector", "hybrid"}:
    raise ValueError("ADVANCED_AGENT_MEMORY_SEARCH_STRATEGY must be keyword, vector, or hybrid")


### Resolve required credentials without exposing them

Oracle credentials are always required. An OpenAI key is required when
embeddings, report synthesis, or vector/hybrid Agent Memory search uses
OpenAI. E2B remains a separate execution-isolation choice and can be disabled
with `ADVANCED_SANDBOX_LIVE=0`.


In [3]:
updates = {}
if not runtime_settings.ora_password:
    password = getpass("Oracle application-user password (hidden): ").strip()
    if not password:
        raise RuntimeError("An Oracle password is required for this notebook")
    updates["ora_password"] = password
needs_openai = (
    runtime_settings.semantic_backend == "openai"
    or runtime_settings.use_model_synthesis
    or AGENT_MEMORY_SEARCH_STRATEGY in {"vector", "hybrid"}
)
if needs_openai and not runtime_settings.openai_api_key:
    api_key = getpass("OpenAI key for embeddings and report drafting (hidden): ").strip()
    if not api_key:
        raise RuntimeError("The selected embedding or synthesis profile requires OPENAI_API_KEY")
    updates["openai_api_key"] = api_key
if REMOTE_SANDBOX_MODE and not runtime_settings.e2b_api_key:
    e2b_key = getpass("E2B key for fresh tool sandboxes (hidden): ").strip()
    if not e2b_key:
        raise RuntimeError("Set E2B_API_KEY or choose ADVANCED_SANDBOX_LIVE=0")
    updates["e2b_api_key"] = e2b_key
runtime_settings = replace(runtime_settings, **updates)
del updates


### Confirm the active profile safely

This diagnostic is intentionally secret-free. It confirms package versions,
the forced Oracle backend, and which optional model and sandbox paths will be
exercised.


In [4]:
print({
    "python": sys.version.split()[0],
    "langgraph": version("langgraph"),
    "langgraph-oracledb": version("langgraph-oracledb"),
    "oracleagentmemory": version("oracleagentmemory"),
    "backend": runtime_settings.backend,
    "semantic_backend": runtime_settings.semantic_backend,
    "model_enabled": runtime_settings.use_model_synthesis,
    "remote_sandbox": bool(runtime_settings.e2b_api_key and REMOTE_SANDBOX_MODE),
    "oracle_password_configured": bool(runtime_settings.ora_password),
})


{'python': '3.12.13', 'langgraph': '1.2.11', 'langgraph-oracledb': '1.0.1', 'oracleagentmemory': '26.6.0', 'backend': 'oracle', 'semantic_backend': 'openai', 'model_enabled': True, 'remote_sandbox': True, 'oracle_password_configured': True}


### Open one shared Oracle connection pool

Both checkpointing and Agent Memory receive the same application pool. The
pool centralises connection limits while the two libraries keep their own
schemas and data models.


In [5]:
import oracledb
from langgraph_oracledb.checkpoint.oracle import OracleSaver
from oracleagentmemory.core import (
    MemoryExtractionConfig,
    OracleAgentMemory,
    OracleDBMemoryStore,
    SchemaPolicy,
    SearchIndexSyncMode,
    SearchStrategy,
)

oracle_pool = oracledb.create_pool(
    **runtime_settings.oracle_connect_kwargs(),
    min=runtime_settings.ora_pool_min,
    max=max(4, runtime_settings.ora_pool_max),
    increment=1,
)


### Install the durable LangGraph checkpointer

`OracleSaver` persists graph state and the next executable node for each
thread. Calling `setup()` creates its database objects when needed. Setting
the JSON threshold to zero keeps checkpoint payloads in Oracle's native JSON
representation.


In [6]:
checkpointer = OracleSaver(oracle_pool, json_size_threshold_mb=0.0)
checkpointer.setup()


### Build Oracle Agent Memory on the same durable substrate

Keyword search needs an Oracle Text index; vector and hybrid search need an
embedder. Keep the requested strategy so the next cell can report when an
older, keyword-only schema requires a safe compatibility mode.


In [7]:
requested_strategy = SearchStrategy(AGENT_MEMORY_SEARCH_STRATEGY)
strategy = requested_strategy
oam_embedder = None
if strategy in {SearchStrategy.VECTOR, SearchStrategy.HYBRID}:
    from oracleagentmemory.core.embedders import Embedder

    oam_embedder = Embedder(
        model=f"openai/{runtime_settings.openai_embed_model}",
        api_key=runtime_settings.openai_api_key,
        embedding_dimension=runtime_settings.embedding_dimensions,
        max_input_tokens=AGENT_MEMORY_EMBED_MAX_TOKENS,
    )

store_options = {
    "embedder": oam_embedder,
    "pool": oracle_pool,
    "memory_store_id": AGENT_MEMORY_STORE_ID,
    "search_strategy": strategy,
}
if strategy in {SearchStrategy.KEYWORD, SearchStrategy.HYBRID}:
    store_options["search_index_sync"] = SearchIndexSyncMode.ON_COMMIT


### Open the memory store and verify the durable providers

A provisioned Agent Memory schema is reused when present; otherwise the first
run creates it. Some existing schemas were created without local vector
columns. `CREATE_IF_NECESSARY` cannot retrofit those columns, so this cell
detects that exact condition and safely reopens the same Oracle schema with
keyword search. It never recreates the schema or deletes stored memories.


In [8]:
compatibility_note = None

try:
    memory_store = OracleDBMemoryStore(
        **store_options, schema_policy=SchemaPolicy.REQUIRE_EXISTING
    )
except Exception:
    try:
        memory_store = OracleDBMemoryStore(
            **store_options, schema_policy=SchemaPolicy.CREATE_IF_NECESSARY
        )
    except ValueError as exc:
        schema_has_no_vectors = (
            "without local vector storage" in str(exc)
            and requested_strategy
            in {SearchStrategy.VECTOR, SearchStrategy.HYBRID}
        )
        if not schema_has_no_vectors:
            raise

        strategy = SearchStrategy.KEYWORD
        store_options = {
            "embedder": None,
            "pool": oracle_pool,
            "memory_store_id": AGENT_MEMORY_STORE_ID,
            "search_strategy": strategy,
            "search_index_sync": SearchIndexSyncMode.ON_COMMIT,
        }
        memory_store = OracleDBMemoryStore(
            **store_options,
            schema_policy=SchemaPolicy.REQUIRE_EXISTING,
        )
        compatibility_note = (
            "The existing managed schema has no local vector storage; "
            "using Oracle keyword search without recreating or deleting data."
        )

agent_memory = OracleAgentMemory(
    store=memory_store,
    memory_extraction_config=MemoryExtractionConfig(extract_memories=False),
)
persistence_status = {
    "checkpointer": type(checkpointer).__name__,
    "agent_memory": type(agent_memory).__name__,
    "memory_substrate": type(memory_store).__name__,
    "requested_search_strategy": requested_strategy.value,
    "active_search_strategy": strategy.value,
    "compatibility_note": compatibility_note,
    "durable": True,
}

pprint(persistence_status)
assert persistence_status["checkpointer"] == "OracleSaver"
assert persistence_status["agent_memory"] == "OracleAgentMemory"
assert persistence_status["memory_substrate"] == "OracleDBMemoryStore"
assert persistence_status["durable"] is True


{'active_search_strategy': 'keyword',
 'agent_memory': 'OracleAgentMemory',
 'checkpointer': 'OracleSaver',
 'compatibility_note': 'The existing managed schema has no local vector '
                       'storage; using Oracle keyword search without '
                       'recreating or deleting data.',
 'durable': True,
 'memory_substrate': 'OracleDBMemoryStore',
 'requested_search_strategy': 'vector'}


### Record durable operational facts

`remember` gives every event a stable ID and stores the canonical JSON event
through Oracle Agent Memory. Because this notebook is Oracle-only, failure to
write memory is an error rather than a reason to silently append to a Python
list.


In [9]:
def remember(thread_id, event_type, payload):
    event = {
        "event_id": uuid.uuid4().hex,
        "thread_id": thread_id,
        "event_type": event_type,
        "payload": payload,
    }
    agent_memory.add_memory(
        json.dumps(event, sort_keys=True, default=str),
        memory_type="fact",
        memory_id=event["event_id"],
        agent_id="supplier-workflow-agent",
        metadata={
            "application": "supplier-workflow",
            "thread_id": thread_id,
            "event_type": event_type,
        },
    )
    return event


## Part II — write and register the toolbox and Skillbox

### The toolbox: executable Python plus database contracts

A tool has two halves. The Python function is executable and stays in the
process registry. The database row contains what a model needs to select it:
name, description, category, JSON Schema, and an embedding of an enriched
description. `register_tool` performs both registrations visibly below.

Semantic retrieval keeps prompt size tied to the current objective rather
than the total number of tools. The query vector is compared with catalog
vectors and only top-k contracts are bound. An HNSW index is requested in
Oracle; a constrained database may use labelled exact cosine without changing
retrieval semantics.

```mermaid
flowchart LR
  Functions[Python tool functions] --> Register[register_tool]
  Schemas[JSON contracts] --> Register
  Register --> Callables[TOOL_REGISTRY]
  Register --> Rows[(Toolbox rows and vectors)]
  Intent[Current objective] --> Query[Query embedding]
  Query --> Rows
  Rows --> TopK[Top-k contracts]
  TopK --> Bind[Authorised tool subset]
  Bind --> Callables
```

### The Skillbox: complete procedures with progressive disclosure

A Skill is a full `SKILL.md` document, not a label for prose hidden elsewhere.
The row includes its SHA-256 version, required tools, source lineage, body,
and semantic vector. Retrieval first returns a small name/description
manifest. The full body loads only after a procedure is selected.


In [10]:
TOOL_TABLE = "NB_AGENT_TOOLS"
SKILL_TABLE = "NB_AGENT_SKILLS"
WORKFLOW_TABLE = "NB_WORKFLOW_RECIPES"
EMBEDDING_DIMENSIONS = runtime_settings.embedding_dimensions
TOOL_REGISTRY = {}
_openai_client = None

def hash_embedding(text, dimensions=EMBEDDING_DIMENSIONS):
    vector = [0.0] * dimensions
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    for token in tokens + [f"{a}_{b}" for a, b in zip(tokens, tokens[1:])]:
        digest = hashlib.sha256(token.encode()).digest()
        bucket = int.from_bytes(digest[:4], "big") % dimensions
        vector[bucket] += -1.0 if digest[4] & 1 else 1.0
    norm = math.sqrt(sum(value * value for value in vector)) or 1.0
    return [value / norm for value in vector]

def embed_text(text):
    global _openai_client
    if runtime_settings.semantic_backend != "openai":
        return hash_embedding(text)
    if _openai_client is None:
        from openai import OpenAI
        _openai_client = OpenAI(api_key=runtime_settings.openai_api_key)
    request = {"model": runtime_settings.openai_embed_model, "input": text}
    if runtime_settings.openai_embed_model.startswith("text-embedding-3"):
        request["dimensions"] = EMBEDDING_DIMENSIONS
    return [float(value) for value in _openai_client.embeddings.create(**request).data[0].embedding]


### Normalise Oracle values and DDL outcomes

Oracle may return JSON directly, as text, or through a LOB object, so
`json_value` reduces those representations to ordinary Python values.
`run_ddl` treats “already exists” as success and reports when a vector index
is unavailable; every other database error remains visible.


In [11]:
def oracle_error_code(exc):
    detail = exc.args[0] if getattr(exc, "args", ()) else None
    return int(getattr(detail, "code", 0) or 0) or None

def json_value(value):
    if hasattr(value, "read"):
        value = value.read()
    if isinstance(value, (str, bytes, bytearray)):
        return json.loads(value)
    return value

def run_ddl(statement, vector_index=False):
    with oracle_pool.acquire() as connection:
        with connection.cursor() as cursor:
            try:
                cursor.execute(statement)
            except Exception as exc:
                if oracle_error_code(exc) == 955:
                    return True
                if vector_index and oracle_error_code(exc) == 51962:
                    return False
                raise
    return True


### Create the Toolbox, Skillbox, and workflow-recipe tables

The DDL is idempotent: rerunning the notebook reuses existing objects. HNSW
indexes accelerate approximate cosine search when the database permits them.
If index creation is unavailable, retrieval still executes exact cosine
distance inside Oracle; data never falls back to process memory.


In [12]:
run_ddl(f'''CREATE TABLE {TOOL_TABLE} (
    name VARCHAR2(120) PRIMARY KEY,
    description VARCHAR2(600) NOT NULL,
    category VARCHAR2(60) NOT NULL,
    tool_schema JSON NOT NULL,
    augmented_description CLOB NOT NULL,
    embedding VECTOR({EMBEDDING_DIMENSIONS}, FLOAT32) NOT NULL,
    updated_at TIMESTAMP DEFAULT SYSTIMESTAMP NOT NULL)''')
tool_hnsw = run_ddl(f'''CREATE VECTOR INDEX NB_TOOLS_HNSW
    ON {TOOL_TABLE}(embedding)
    ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE
    WITH TARGET ACCURACY 95''', vector_index=True)

run_ddl(f'''CREATE TABLE {SKILL_TABLE} (
    name VARCHAR2(120) PRIMARY KEY,
    description VARCHAR2(600) NOT NULL,
    sha VARCHAR2(64) NOT NULL,
    skill_md CLOB NOT NULL,
    tools_used JSON NOT NULL,
    source_workflow_id VARCHAR2(64),
    embedding VECTOR({EMBEDDING_DIMENSIONS}, FLOAT32) NOT NULL,
    updated_at TIMESTAMP DEFAULT SYSTIMESTAMP NOT NULL)''')
skill_hnsw = run_ddl(f'''CREATE VECTOR INDEX NB_SKILLS_HNSW
    ON {SKILL_TABLE}(embedding)
    ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE
    WITH TARGET ACCURACY 95''', vector_index=True)


### Create the workflow-learning table

Recipes are separate from Skills: they count raw recurring trajectories
until promotion policy decides one has earned a reusable procedure. The
status output also records whether Toolbox and Skillbox search use HNSW or
exact cosine inside Oracle.


In [13]:
run_ddl(f'''CREATE TABLE {WORKFLOW_TABLE} (
    recipe_id VARCHAR2(64) PRIMARY KEY,
    fingerprint VARCHAR2(64) UNIQUE NOT NULL,
    intent VARCHAR2(1000) NOT NULL,
    trajectory JSON NOT NULL,
    tools_used JSON NOT NULL,
    occurrences NUMBER DEFAULT 0 NOT NULL,
    successes NUMBER DEFAULT 0 NOT NULL,
    promoted CHAR(1) DEFAULT 'N' NOT NULL,
    embedding VECTOR({EMBEDDING_DIMENSIONS}, FLOAT32) NOT NULL,
    updated_at TIMESTAMP DEFAULT SYSTIMESTAMP NOT NULL)''')

print({
    "toolbox": TOOL_TABLE,
    "skillbox": SKILL_TABLE,
    "dimensions": EMBEDDING_DIMENSIONS,
    "tool_hnsw": tool_hnsw,
    "skill_hnsw": skill_hnsw,
    "search_mode": "HNSW" if tool_hnsw and skill_hnsw else "exact cosine in Oracle",
})


{'toolbox': 'NB_AGENT_TOOLS', 'skillbox': 'NB_AGENT_SKILLS', 'dimensions': 1536, 'tool_hnsw': True, 'skill_hnsw': True, 'search_mode': 'HNSW'}


### Register executable tools and their database contracts

`register_tool` keeps the trusted Python callable in the local registry and
upserts its model-facing schema, enriched description, and vector into Oracle.
A model can discover a contract, but execution still resolves through the
explicit callable registry.


In [14]:
def register_tool(name, fn, description, parameters, *, category, synonyms=(), when_to_use=""):
    schema = {
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": parameters,
            "required": list(parameters),
            "additionalProperties": False,
        },
    }
    enriched = (
        f"TOOL {name}: {description}\ncategory: {category}\n"
        f"synonyms: {', '.join(synonyms)}\nuse when: {when_to_use}"
    )
    vector = embed_text(enriched)
    TOOL_REGISTRY[name] = {"fn": fn, "schema": schema}
    with oracle_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(f'''MERGE INTO {TOOL_TABLE} d
                USING (SELECT :tool_name name FROM dual) s ON (d.name=s.name)
                WHEN MATCHED THEN UPDATE SET description=:description,
                    category=:category, tool_schema=:tool_schema,
                    augmented_description=:augmented, embedding=:embedding,
                    updated_at=SYSTIMESTAMP
                WHEN NOT MATCHED THEN INSERT
                    (name,description,category,tool_schema,augmented_description,embedding)
                    VALUES (:tool_name,:description,:category,:tool_schema,:augmented,:embedding)''', {
                "tool_name": name,
                "description": description,
                "category": category,
                "tool_schema": json.dumps(schema, sort_keys=True),
                "augmented": enriched,
                "embedding": array("f", vector),
            })
        connection.commit()
    return schema


### Retrieve only the relevant tool contracts

Retrieval embeds the current objective and asks Oracle for the closest rows.
The callable implementations remain in `TOOL_REGISTRY`; only JSON contracts
and metadata cross the semantic retrieval boundary.


In [15]:
def retrieve_tools(query, k=6):
    query_vector = embed_text(query)
    fetch_mode = "FETCH APPROX FIRST" if tool_hnsw else "FETCH FIRST"
    with oracle_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(f'''SELECT name,description,category,tool_schema,
                VECTOR_DISTANCE(embedding,:query_vector,COSINE) distance
                FROM {TOOL_TABLE} ORDER BY distance {fetch_mode} {int(k)} ROWS ONLY''',
                {"query_vector": array("f", query_vector)})
            rows = cursor.fetchall()
    return [{
        "name": str(row[0]), "description": str(row[1]), "category": str(row[2]),
        "tool_schema": json_value(row[3]),
        "distance": round(float(row[4]), 6),
    } for row in rows]


### Save complete, versioned Skills

Each Skill row stores the whole `SKILL.md`, its SHA-256 digest, required tool
names, optional workflow lineage, and an embedding. `MERGE` makes registration
repeatable while preserving one current row per Skill name.


In [16]:
def save_skill(name, description, skill_md, tools_used, source_workflow_id=None):
    sha = hashlib.sha256(skill_md.encode()).hexdigest()
    vector = embed_text(f"SKILL {name}: {description}\ntools: {', '.join(tools_used)}\n{skill_md}")
    with oracle_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(f'''MERGE INTO {SKILL_TABLE} d
                USING (SELECT :skill_name name FROM dual) s ON (d.name=s.name)
                WHEN MATCHED THEN UPDATE SET description=:description,sha=:sha,
                    skill_md=:skill_md,tools_used=:tools_used,
                    source_workflow_id=:source_workflow_id,embedding=:embedding,
                    updated_at=SYSTIMESTAMP
                WHEN NOT MATCHED THEN INSERT
                    (name,description,sha,skill_md,tools_used,source_workflow_id,embedding)
                    VALUES (:skill_name,:description,:sha,:skill_md,:tools_used,
                    :source_workflow_id,:embedding)''', {
                "skill_name": name, "description": description, "sha": sha,
                "skill_md": skill_md, "tools_used": json.dumps(list(tools_used)),
                "source_workflow_id": source_workflow_id,
                "embedding": array("f", vector),
            })
        connection.commit()
    return sha


### Apply progressive disclosure to Skills

`retrieve_skills` returns a compact manifest for ranking. `load_skill` performs
a second Oracle read only for selected procedures, returning the complete body
and its lineage. This keeps large instructions out of context until needed.


In [17]:
def retrieve_skills(query, k=3):
    query_vector = embed_text(query)
    fetch_mode = "FETCH APPROX FIRST" if skill_hnsw else "FETCH FIRST"
    with oracle_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(f'''SELECT name,description,
                VECTOR_DISTANCE(embedding,:query_vector,COSINE) distance
                FROM {SKILL_TABLE} ORDER BY distance {fetch_mode} {int(k)} ROWS ONLY''',
                {"query_vector": array("f", query_vector)})
            rows = cursor.fetchall()
    return [{"name": str(row[0]), "description": str(row[1]),
             "distance": round(float(row[2]), 6)} for row in rows]

def load_skill(name):
    with oracle_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(f'''SELECT name,description,sha,skill_md,tools_used,source_workflow_id
                FROM {SKILL_TABLE} WHERE name=:skill_name''', {"skill_name": name})
            row = cursor.fetchone()
    if row is None:
        return {"error": "no such skill"}
    body = row[3].read() if hasattr(row[3], "read") else str(row[3])
    return {"name": str(row[0]), "description": str(row[1]), "sha": str(row[2]),
            "skill_md": body, "tools_used": json_value(row[4]),
            "source_workflow_id": row[5]}


### Define three governed teaching cases

The fixtures make each route observable: Northstar requires a full review,
Bluebird qualifies for the short path, and Contoso must stop for missing
evidence. Copying a returned case prevents a tool from mutating the fixture.


In [18]:
CASES = {
    "northstar": {
        "supplier_id": "northstar", "supplier": "Northstar Textiles",
        "risk_class": "high", "annual_spend_usd": 4_800_000,
        "required_documents": ["restricted_substance_certificate", "factory_audit"],
        "documents": {"restricted_substance_certificate": {"age_days": 401}, "factory_audit": {"age_days": 120}},
        "sanctions_matches": 0, "open_audit_actions": 1,
    },
    "bluebird": {
        "supplier_id": "bluebird", "supplier": "Bluebird Components",
        "risk_class": "standard", "annual_spend_usd": 240_000,
        "required_documents": ["restricted_substance_certificate", "factory_audit"],
        "documents": {"restricted_substance_certificate": {"age_days": 90}, "factory_audit": {"age_days": 45}},
        "sanctions_matches": 0, "open_audit_actions": 0,
    },
    "contoso": {
        "supplier_id": "contoso", "supplier": "Contoso Fibres",
        "risk_class": "standard", "annual_spend_usd": 310_000,
        "required_documents": ["restricted_substance_certificate", "factory_audit"],
        "documents": {"factory_audit": {"age_days": 70}},
        "sanctions_matches": 0, "open_audit_actions": 0,
    },
}


### Write independent evidence tools

These functions read authoritative case fields and return small structured
findings. They do not choose the workflow route or publication authority;
those policy decisions remain visible in later graph functions.


In [19]:
def load_supplier_case(supplier_id):
    if supplier_id not in CASES:
        return {"error": "unknown supplier"}
    return json.loads(json.dumps(CASES[supplier_id]))

def validate_document_freshness(case):
    certificate = case.get("documents", {}).get("restricted_substance_certificate")
    age_days = certificate.get("age_days") if certificate else None
    return {"check": "freshness", "age_days": age_days,
            "expired": age_days is None or age_days > 365,
            "blocking": age_days is None}

def screen_supplier_sanctions(case):
    matches = int(case.get("sanctions_matches", 0))
    return {"check": "sanctions", "matches": matches,
            "clear": matches == 0, "blocking": matches > 0}

def review_open_audits(case):
    open_actions = int(case.get("open_audit_actions", 0))
    return {"check": "audit", "open_actions": open_actions,
            "clear": open_actions == 0, "blocking": False}


### Convert verified findings into action

Risk scoring is deterministic and inspectable. Remediation translates
findings into required actions, while publication binds the exact report and
approval to a stable digest.


In [20]:
def calculate_supplier_risk(case, checks):
    score = 10
    if case.get("risk_class") == "high": score += 25
    if case.get("annual_spend_usd", 0) >= 1_000_000: score += 15
    if any(item.get("check") == "freshness" and item.get("expired") for item in checks): score += 30
    if any(item.get("check") == "audit" and item.get("open_actions", 0) for item in checks): score += 20
    if any(item.get("check") == "sanctions" and not item.get("clear", True) for item in checks): score = 100
    return {"score": min(score, 100), "blocking": any(item.get("blocking") for item in checks)}

def build_remediation_plan(case, checks, risk):
    actions = []
    if any(item.get("expired") for item in checks): actions.append("obtain a current restricted-substance certificate")
    if any(item.get("open_actions", 0) for item in checks): actions.append("close the outstanding factory-audit action")
    if risk.get("blocking"): actions.append("hold purchasing authority until the blocking finding clears")
    return {"supplier_id": case["supplier_id"], "risk_score": risk["score"], "actions": actions}

def publish_report(report, approval):
    canonical = json.dumps({"report": report, "approval": approval}, sort_keys=True, separators=(",", ":"))
    return {"publication_id": "PUB-" + hashlib.sha256(canonical.encode()).hexdigest()[:16],
            "digest": hashlib.sha256(canonical.encode()).hexdigest(), "published": True}


### Attach contracts and verify semantic discovery

Each tuple pairs a callable with its JSON parameters, category, synonyms, and
usage guidance. Registration writes all seven contracts to Oracle; the probe
then confirms that one broad compliance objective can retrieve them again.


In [21]:
TOOL_DEFINITIONS = [
    ("load_supplier_case", load_supplier_case, "Load one governed supplier case.",
     {"supplier_id": {"type": "string"}}, "intake", ("supplier lookup",), "at the start of every review"),
    ("validate_document_freshness", validate_document_freshness, "Check required certificate age and missing evidence.",
     {"case": {"type": "object"}}, "compliance", ("expiry check",), "when evidence freshness affects renewal"),
    ("screen_supplier_sanctions", screen_supplier_sanctions, "Read the authoritative sanctions-screen result.",
     {"case": {"type": "object"}}, "compliance", ("watch-list check",), "for every supplier publication decision"),
    ("review_open_audits", review_open_audits, "Count unresolved factory-audit actions.",
     {"case": {"type": "object"}}, "compliance", ("audit remediation",), "for full high-risk reviews"),
    ("calculate_supplier_risk", calculate_supplier_risk, "Calculate the policy risk score from verified checks.",
     {"case": {"type": "object"}, "checks": {"type": "array"}}, "risk", ("risk score",), "after independent checks join"),
    ("build_remediation_plan", build_remediation_plan, "Create actions for expired, open, or blocking findings.",
     {"case": {"type": "object"}, "checks": {"type": "array"}, "risk": {"type": "object"}}, "remediation", ("corrective action",), "when score or findings require remediation"),
    ("publish_report", publish_report, "Create a digest-bound publication record.",
     {"report": {"type": "object"}, "approval": {"type": "object"}}, "publication", ("release report",), "only after the legal authority path completes"),
]

for name, fn, description, parameters, category, synonyms, when_to_use in TOOL_DEFINITIONS:
    register_tool(name, fn, description, parameters, category=category,
                  synonyms=synonyms, when_to_use=when_to_use)

probe_tools = retrieve_tools(
    "load a supplier, validate an expired certificate, screen sanctions, score risk, remediate, and publish",
    k=7,
)
pprint([{"rank": index + 1, "name": row["name"], "distance": row["distance"]}
        for index, row in enumerate(probe_tools)])
assert len(TOOL_REGISTRY) == 7 and len(probe_tools) == 7
assert all("parameters" in row["tool_schema"] for row in probe_tools)


[{'distance': 0.382186, 'name': 'screen_supplier_sanctions', 'rank': 1},
 {'distance': 0.522884, 'name': 'calculate_supplier_risk', 'rank': 2},
 {'distance': 0.522975, 'name': 'load_supplier_case', 'rank': 3},
 {'distance': 0.55757, 'name': 'validate_document_freshness', 'rank': 4},
 {'distance': 0.591969, 'name': 'build_remediation_plan', 'rank': 5},
 {'distance': 0.625365, 'name': 'review_open_audits', 'rank': 6},
 {'distance': 0.70334, 'name': 'publish_report', 'rank': 7}]


### Author complete procedures as `SKILL.md` documents

The frontmatter names each procedure and its allowed tools. The body contains
the operational steps. Storing the entire document makes the procedure
portable, inspectable, and hash-verifiable.


In [22]:
HIGH_RISK_SKILL = '''---
name: high-risk-supplier-review
description: Run independent evidence, sanctions, and audit checks before scoring a material or high-risk supplier.
tools: [load_supplier_case, validate_document_freshness, screen_supplier_sanctions, review_open_audits, calculate_supplier_risk, build_remediation_plan, publish_report]
---
## Steps
1. Load the governed supplier record.
2. Run freshness, sanctions, and audit checks independently and in parallel.
3. Join only complete check results, then calculate the policy score.
4. Build remediation for an expired certificate, open audit action, or blocking finding.
5. Require a named decision before publication when score is at least 70 or a finding blocks.
6. Publish the exact approved report and verify its digest.
'''
STANDARD_SKILL = '''---
name: standard-supplier-renewal
description: Use a shorter verified path for complete, standard-risk, low-materiality supplier renewals.
tools: [load_supplier_case, validate_document_freshness, screen_supplier_sanctions, calculate_supplier_risk, publish_report]
---
## Steps
1. Confirm required evidence exists.
2. Check freshness and sanctions status.
3. Calculate risk and use policy publication only when no blocking condition exists.
'''
EVIDENCE_GAP_SKILL = '''---
name: supplier-evidence-gap
description: Stop a supplier review safely when a required document is absent.
tools: [load_supplier_case]
---
## Steps
1. List required documents that are absent.
2. Request those documents and stop the decision path.
3. Never infer or fabricate a missing control result.
'''


### Persist and retrieve the initial Skillbox

The first three calls version and upsert the canonical procedures. The probe
demonstrates progressive disclosure by retrieving a compact manifest first
and loading full documents second.


In [23]:
save_skill("high-risk-supplier-review",
           "Run independent evidence, sanctions, and audit checks before scoring a material or high-risk supplier.",
           HIGH_RISK_SKILL, [item[0] for item in TOOL_DEFINITIONS])
save_skill("standard-supplier-renewal",
           "Use a shorter verified path for complete, standard-risk, low-materiality supplier renewals.",
           STANDARD_SKILL, ["load_supplier_case", "validate_document_freshness", "screen_supplier_sanctions", "calculate_supplier_risk", "publish_report"])
save_skill("supplier-evidence-gap",
           "Stop a supplier review safely when a required document is absent.",
           EVIDENCE_GAP_SKILL, ["load_supplier_case"])

skill_manifest = retrieve_skills("review supplier evidence, risk, sanctions, audits, and renewal", k=3)
full_skills = [load_skill(row["name"]) for row in skill_manifest]
pprint({"manifest": skill_manifest,
        "loaded": [{"name": row["name"], "sha": row["sha"][:12]} for row in full_skills]})
assert all(len(row["sha"]) == 64 and row["skill_md"].startswith("---") for row in full_skills)


{'loaded': [{'name': 'high-risk-supplier-review', 'sha': '1fc0a37aa7c0'},
            {'name': 'standard-supplier-renewal', 'sha': 'f15a28cf5a49'},
            {'name': 'supplier-evidence-gap', 'sha': '4e8cf52372aa'}],
 'manifest': [{'description': 'Run independent evidence, sanctions, and audit '
                              'checks before scoring a material or high-risk '
                              'supplier.',
               'distance': 0.365328,
               'name': 'high-risk-supplier-review'},
              {'description': 'Use a shorter verified path for complete, '
                              'standard-risk, low-materiality supplier '
                              'renewals.',
               'distance': 0.408275,
               'name': 'standard-supplier-renewal'},
              {'description': 'Stop a supplier review safely when a required '
                              'document is absent.',
               'distance': 0.510565,
               'name': 'supplier-eviden

## Part III — execute retrieved tools through a visible sandbox boundary

`execute_registered_tool` enforces four checks before a result can enter graph
state: the tool name must exist in the Python registry, it must be authorised
by a loaded Skill, its portable implementation runs with an empty environment,
and the host callable must independently reproduce the result. A fresh E2B
provider is created and closed for each live invocation. The explicit local
subprocess profile demonstrates mechanics but reports that remote isolation
was not proven.

The input, execution policy, output, digests, and teardown evidence form one
envelope. This makes “the tool succeeded” an inspectable claim rather than a
sentence generated by the model.

```mermaid
sequenceDiagram
  participant N as Graph node
  participant B as Retrieved tool binding
  participant S as Fresh sandbox
  participant H as Host verifier
  N->>B: tool name + JSON arguments
  B->>B: check Skill authorisation
  B->>S: portable program + input · env empty
  S-->>B: JSON result + exit status
  B->>H: recompute with registered callable
  H-->>B: exact match or fail closed
  B-->>N: input/execution/output envelope
  B->>S: close disposable session
```


### Create a portable sandbox program

Sandboxes cannot call trusted in-process functions directly, so the small
deterministic implementations are repeated as source. The host versions
remain independent verifiers: any drift between the two copies fails closed.


In [24]:
# Portable functions are intentionally repeated as sandbox source. The host
# functions above remain the independent verifier, so drift fails closed.
SANDBOX_LIBRARY = "import hashlib, json\nCASES = " + repr(CASES) + r'''
def load_supplier_case(supplier_id):
    if supplier_id not in CASES: return {"error": "unknown supplier"}
    return json.loads(json.dumps(CASES[supplier_id]))
def validate_document_freshness(case):
    certificate = case.get("documents", {}).get("restricted_substance_certificate")
    age_days = certificate.get("age_days") if certificate else None
    return {"check":"freshness","age_days":age_days,"expired":age_days is None or age_days > 365,"blocking":age_days is None}
def screen_supplier_sanctions(case):
    matches = int(case.get("sanctions_matches", 0))
    return {"check":"sanctions","matches":matches,"clear":matches == 0,"blocking":matches > 0}
def review_open_audits(case):
    open_actions = int(case.get("open_audit_actions", 0))
    return {"check":"audit","open_actions":open_actions,"clear":open_actions == 0,"blocking":False}
def calculate_supplier_risk(case, checks):
    score = 10
    if case.get("risk_class") == "high": score += 25
    if case.get("annual_spend_usd", 0) >= 1000000: score += 15
    if any(x.get("check") == "freshness" and x.get("expired") for x in checks): score += 30
    if any(x.get("check") == "audit" and x.get("open_actions", 0) for x in checks): score += 20
    if any(x.get("check") == "sanctions" and not x.get("clear", True) for x in checks): score = 100
    return {"score":min(score,100),"blocking":any(x.get("blocking") for x in checks)}
def build_remediation_plan(case, checks, risk):
    actions=[]
    if any(x.get("expired") for x in checks): actions.append("obtain a current restricted-substance certificate")
    if any(x.get("open_actions", 0) for x in checks): actions.append("close the outstanding factory-audit action")
    if risk.get("blocking"): actions.append("hold purchasing authority until the blocking finding clears")
    return {"supplier_id":case["supplier_id"],"risk_score":risk["score"],"actions":actions}
def publish_report(report, approval):
    canonical=json.dumps({"report":report,"approval":approval},sort_keys=True,separators=(",",":"))
    return {"publication_id":"PUB-"+hashlib.sha256(canonical.encode()).hexdigest()[:16],"digest":hashlib.sha256(canonical.encode()).hexdigest(),"published":True}
TOOLS = {name: value for name, value in globals().items() if callable(value) and name in {
    "load_supplier_case","validate_document_freshness","screen_supplier_sanctions","review_open_audits",
    "calculate_supplier_risk","build_remediation_plan","publish_report"}}
'''


### Run code through disposable isolation

This helper selects remote E2B or the labelled local subprocess profile,
forwards an empty environment, captures execution evidence, and closes the
disposable remote provider in `finally`. Capability checks and host
recomputation stay in the next layer.


In [25]:
def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)

def run_sandbox_program(program, tool_name):
    provider = None
    try:
        if runtime_settings.e2b_api_key and REMOTE_SANDBOX_MODE:
            from memorizz.sandbox import E2BSandboxProvider
            provider = E2BSandboxProvider(
                api_key=runtime_settings.e2b_api_key,
                session_timeout=runtime_settings.e2b_session_timeout,
                max_execution_timeout=runtime_settings.e2b_execution_timeout,
                allow_internet_access=False,
            )
            execution = provider.execute_code(
                program, language="python",
                timeout=runtime_settings.e2b_execution_timeout,
                envs={},
            )
            stdout = "\n".join(execution.stdout or [])
            exit_code = execution.exit_code
            provider_name = "MemoRizz E2BSandboxProvider"
            isolation_proven = True
        else:
            execution = subprocess.run(
                [sys.executable, "-I", "-c", program],
                capture_output=True, text=True, timeout=30, env={}, check=False,
            )
            stdout, exit_code = execution.stdout, execution.returncode
            provider_name = "isolated local subprocess fallback"
            isolation_proven = False
        if exit_code != 0:
            raise RuntimeError(f"sandbox execution failed for {tool_name}")
        evidence = {
            "provider": provider_name,
            "isolation_proven": isolation_proven,
            "egress_allowed": False,
            "environment_variable_count": 0,
            "timeout_seconds": runtime_settings.e2b_execution_timeout if isolation_proven else 30,
            "session_closed": True,
        }
        return stdout, evidence
    finally:
        if provider is not None:
            provider.close()


### Verify the selected tool and build its evidence envelope

This layer owns capability checks and semantic validation. It builds the
portable program, decodes the sandbox result, recomputes it with the trusted
callable, and combines input, program, and output digests into one record.


In [26]:
def execute_registered_tool(name, arguments, authorised_tools):
    if name not in TOOL_REGISTRY:
        raise KeyError(f"No registered callable for {name}")
    if name != "load_supplier_case" and name not in authorised_tools:
        raise PermissionError(f"{name} was not authorised by a loaded Skill")

    payload = canonical(arguments)
    program = SANDBOX_LIBRARY + (
        f"\narguments=json.loads({payload!r})"
        f"\nresult=TOOLS[{name!r}](**arguments)"
        "\nprint(json.dumps(result,sort_keys=True))\n"
    )
    stdout, execution_evidence = run_sandbox_program(program, name)
    result = json.loads([line for line in stdout.splitlines() if line.strip()][-1])
    expected = TOOL_REGISTRY[name]["fn"](**arguments)
    if canonical(result) != canonical(expected):
        raise RuntimeError(f"host verification failed for {name}")

    return {
        "tool": name,
        "input": {
            "arguments": arguments,
            "digest": hashlib.sha256(payload.encode()).hexdigest(),
        },
        "execution": {
            **execution_evidence,
            "program_digest": hashlib.sha256(program.encode()).hexdigest(),
        },
        "output": {
            "result": result,
            "host_recomputation_matched": True,
            "digest": hashlib.sha256(canonical(result).encode()).hexdigest(),
        },
    }


### Probe the sandbox boundary before building the graph

A single freshness call proves the output envelope has a host match and that
no environment variables were forwarded. Later graph nodes use this same
execution function for every tool call.


In [27]:
sandbox_probe = execute_registered_tool("validate_document_freshness", {"case": CASES["northstar"]}, set(TOOL_REGISTRY))
pprint(sandbox_probe)
assert sandbox_probe["output"]["host_recomputation_matched"]
assert sandbox_probe["execution"]["environment_variable_count"] == 0


{'execution': {'egress_allowed': False,
               'environment_variable_count': 0,
               'isolation_proven': True,
               'program_digest': '6c760683ba1f5afeaa64c94f49ebfaa102b22e3b4a363b3693f72d72a5eef54e',
               'provider': 'MemoRizz E2BSandboxProvider',
               'session_closed': True,
               'timeout_seconds': 30},
 'input': {'arguments': {'case': {'annual_spend_usd': 4800000,
                                  'documents': {'factory_audit': {'age_days': 120},
                                                'restricted_substance_certificate': {'age_days': 401}},
                                  'open_audit_actions': 1,
                                  'required_documents': ['restricted_substance_certificate',
                                                         'factory_audit'],
                                  'risk_class': 'high',
                                  'sanctions_matches': 0,
                                  'supplie

## Part IV — define the graph nodes and criteria in the open

The route functions below are deliberately small. `route_review` checks
required documents, risk class, and a $1 million materiality threshold.
`route_after_score` determines whether remediation is needed.
`route_after_draft` determines whether a named person must decide. These
rules are ordinary Python and can be tested independently.

The full-review node fans out to three nodes in one LangGraph superstep. A
list reducer merges their checks and sandbox envelopes without last-writer
loss. The list-form edge into `score_risk` is the join barrier: scoring does
not begin until every expected branch has completed.

```mermaid
stateDiagram-v2
  [*] --> LoadCase
  LoadCase --> RetrieveCapabilities
  RetrieveCapabilities --> Classify
  Classify --> NeedsEvidence: required document absent
  Classify --> QuickReview: standard and below materiality
  Classify --> FullReview: high risk or material spend
  state FullReview {
    [*] --> Freshness
    [*] --> Sanctions
    [*] --> Audit
    Freshness --> Barrier
    Sanctions --> Barrier
    Audit --> Barrier
  }
  Barrier --> ScoreRisk
  QuickReview --> ScoreRisk
  ScoreRisk --> Remediation: score >= 60 or blocking
  ScoreRisk --> Draft: otherwise
  Remediation --> Draft
  Draft --> HumanApproval: score >= 70 or blocking
  Draft --> AutoApproval: policy permits
  HumanApproval --> Publish: approve
  HumanApproval --> Rejected: reject
  AutoApproval --> Publish
  Publish --> Verify
  Verify --> [*]
  NeedsEvidence --> [*]
  Rejected --> [*]
```


### Define shared graph state and parallel reducers

`WorkflowState` names every value that may cross a node boundary. The three
`Annotated` lists use `operator.add`, allowing parallel branches to append
checks, sandbox evidence, and trajectory labels without overwriting one
another.


In [28]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

class WorkflowState(TypedDict, total=False):
    thread_id: str
    supplier_id: str
    objective: str
    case: dict
    review_mode: str
    missing_documents: list[str]
    skill_manifest: list[dict]
    loaded_skills: list[dict]
    tool_candidates: list[dict]
    authorised_tools: list[str]
    checks: Annotated[list[dict], operator.add]
    sandbox_trace: Annotated[list[dict], operator.add]
    trajectory: Annotated[list[str], operator.add]
    risk: dict
    remediation: dict
    report: dict
    approval: dict
    publication: dict
    verification: dict
    status: str


### Load, retrieve, and classify before doing review work

Intake first loads the governed record, then intersects semantically retrieved
tool contracts with tools allowed by retrieved Skills. Classification is pure
policy: missing evidence wins, followed by high risk or material spend, then
the quick path.


In [29]:
def run_tool_for_state(state, name, arguments):
    return execute_registered_tool(name, arguments, set(state.get("authorised_tools", [])))

def load_case_node(state):
    envelope = execute_registered_tool("load_supplier_case", {"supplier_id": state["supplier_id"]}, set())
    case = envelope["output"]["result"]
    if "error" in case:
        raise ValueError(case["error"])
    return {"case": case, "sandbox_trace": [envelope], "trajectory": ["load_case"]}

def retrieve_capabilities_node(state):
    case = state["case"]
    query = f"{case['risk_class']} risk supplier renewal with spend {case['annual_spend_usd']}; evidence sanctions audits risk remediation publication"
    manifest = retrieve_skills(query, k=3)
    loaded = [load_skill(row["name"]) for row in manifest]
    tools = sorted({tool for skill in loaded for tool in skill["tools_used"]})
    candidates = retrieve_tools(query, k=7)
    bound = sorted(set(tools) & {row["name"] for row in candidates})
    remember(state["thread_id"], "capabilities_retrieved", {"skills": [x["name"] for x in loaded], "tools": bound})
    return {"skill_manifest": manifest, "loaded_skills": loaded,
            "tool_candidates": candidates, "authorised_tools": bound,
            "trajectory": ["retrieve_capabilities"]}

def classify_case_node(state):
    case = state["case"]
    missing = sorted(set(case["required_documents"]) - set(case.get("documents", {})))
    if missing:
        mode = "missing"
    elif case["risk_class"] == "high" or case["annual_spend_usd"] >= 1_000_000:
        mode = "full"
    else:
        mode = "quick"
    remember(state["thread_id"], "case_classified", {"mode": mode, "missing": missing})
    return {"review_mode": mode, "missing_documents": missing,
            "trajectory": [f"classify:{mode}"]}

def route_review(state):
    return state["review_mode"]

def request_documents_node(state):
    remember(state["thread_id"], "evidence_requested", {"missing": state["missing_documents"]})
    return {"status": "needs_evidence", "trajectory": ["request_documents"]}


### Implement the quick path and the full-review branches

The full path has three one-purpose nodes so LangGraph can run them in the
same superstep. The quick node performs only freshness and sanctions checks.
Both paths converge on the same deterministic score, and remediation is
selected only when the score or blocking flag requires it.


In [30]:
def full_review_node(state):
    return {"trajectory": ["full_review_fanout"]}

def freshness_node(state):
    envelope = run_tool_for_state(state, "validate_document_freshness", {"case": state["case"]})
    return {"checks": [envelope["output"]["result"]], "sandbox_trace": [envelope],
            "trajectory": ["check_freshness"]}

def sanctions_node(state):
    envelope = run_tool_for_state(state, "screen_supplier_sanctions", {"case": state["case"]})
    return {"checks": [envelope["output"]["result"]], "sandbox_trace": [envelope],
            "trajectory": ["screen_sanctions"]}

def audit_node(state):
    envelope = run_tool_for_state(state, "review_open_audits", {"case": state["case"]})
    return {"checks": [envelope["output"]["result"]], "sandbox_trace": [envelope],
            "trajectory": ["review_audits"]}

def quick_review_node(state):
    envelopes = [
        run_tool_for_state(state, "validate_document_freshness", {"case": state["case"]}),
        run_tool_for_state(state, "screen_supplier_sanctions", {"case": state["case"]}),
    ]
    return {"checks": [item["output"]["result"] for item in envelopes],
            "sandbox_trace": envelopes, "trajectory": ["quick_review"]}

def score_risk_node(state):
    envelope = run_tool_for_state(state, "calculate_supplier_risk",
                                  {"case": state["case"], "checks": state["checks"]})
    return {"risk": envelope["output"]["result"], "sandbox_trace": [envelope],
            "trajectory": ["score_risk"]}

def route_after_score(state):
    return "remediate" if state["risk"]["score"] >= 60 or state["risk"]["blocking"] else "draft"

def remediation_node(state):
    envelope = run_tool_for_state(state, "build_remediation_plan",
                                  {"case": state["case"], "checks": state["checks"], "risk": state["risk"]})
    return {"remediation": envelope["output"]["result"], "sandbox_trace": [envelope],
            "trajectory": ["build_remediation"]}


### Draft a report from verified evidence

The model may phrase a report, but it receives only verified JSON facts.
The node records the model or deterministic provider and binds the report to
an evidence digest, so later approval refers to an exact artifact.


In [31]:
def draft_text(state):
    facts = {"supplier": state["case"]["supplier"], "checks": state["checks"],
             "risk": state["risk"], "remediation": state.get("remediation", {})}
    if runtime_settings.use_model_synthesis and runtime_settings.openai_api_key:
        from openai import OpenAI
        response = OpenAI(api_key=runtime_settings.openai_api_key).responses.create(
            model=runtime_settings.openai_model,
            input="Draft a concise supplier-compliance report using only this verified JSON: " + canonical(facts),
        )
        return response.output_text, runtime_settings.openai_model
    summary = (f"{facts['supplier']} received risk score {facts['risk']['score']}. "
               f"Verified checks: {canonical(facts['checks'])}. "
               f"Remediation: {canonical(facts['remediation'])}.")
    return summary, "deterministic labelled fallback"

def draft_report_node(state):
    text, provider = draft_text(state)
    report = {"supplier_id": state["supplier_id"], "text": text,
              "risk_score": state["risk"]["score"], "blocking": state["risk"]["blocking"],
              "model_provider": provider,
              "evidence_digest": hashlib.sha256(canonical(state["checks"]).encode()).hexdigest()}
    remember(state["thread_id"], "report_drafted", {"risk_score": report["risk_score"], "provider": provider})
    return {"report": report, "trajectory": ["draft_report"]}


### Enforce authority and verify terminal outcomes

Ordinary clear cases receive a named policy decision; high scores or blocking
findings pause at `interrupt()`. Publication remains a registered tool call,
rejection becomes an auditable terminal state, and the final node verifies
the publication digest before declaring success.


In [32]:
def route_after_draft(state):
    return "human" if state["risk"]["score"] >= 70 or state["risk"]["blocking"] else "automatic"

def human_approval_node(state):
    decision = interrupt({"supplier": state["case"]["supplier"],
                          "risk_score": state["risk"]["score"],
                          "report_digest": hashlib.sha256(canonical(state["report"]).encode()).hexdigest(),
                          "question": "Approve this exact report for publication?"})
    return {"approval": dict(decision), "trajectory": ["human_approval"]}

def route_human_decision(state):
    return "publish" if state["approval"].get("approved") else "reject"

def automatic_approval_node(state):
    return {"approval": {"approved": True, "decided_by": "policy:standard-low-risk-v1"},
            "trajectory": ["automatic_approval"]}

def publish_node(state):
    envelope = run_tool_for_state(state, "publish_report",
                                  {"report": state["report"], "approval": state["approval"]})
    return {"publication": envelope["output"]["result"], "sandbox_trace": [envelope],
            "status": "published", "trajectory": ["publish"]}

def reject_node(state):
    remember(state["thread_id"], "publication_rejected", state["approval"])
    return {"status": "rejected", "trajectory": ["reject"]}

def verify_node(state):
    verified = bool(state["publication"].get("published") and len(state["publication"].get("digest", "")) == 64)
    if not verified: raise RuntimeError("publication verification failed")
    result = {"verified": True, "publication_id": state["publication"]["publication_id"]}
    remember(state["thread_id"], "publication_verified", result)
    return {"verification": result, "status": "verified", "trajectory": ["verify"]}


### Register nodes by their operational names

The mapping keeps graph labels separate from Python function names. These
labels become checkpoint locations, Mermaid diagram nodes, and trajectory
entries learners can inspect later.


In [33]:
builder = StateGraph(WorkflowState)
for name, fn in {
    "load_case": load_case_node,
    "retrieve_capabilities": retrieve_capabilities_node,
    "classify_case": classify_case_node,
    "request_documents": request_documents_node,
    "full_review": full_review_node,
    "check_freshness": freshness_node,
    "screen_sanctions": sanctions_node,
    "review_audits": audit_node,
    "quick_review": quick_review_node,
    "score_risk": score_risk_node,
    "build_remediation": remediation_node,
    "draft_report": draft_report_node,
    "human_approval": human_approval_node,
    "automatic_approval": automatic_approval_node,
    "publish": publish_node,
    "reject": reject_node,
    "verify": verify_node,
}.items():
    builder.add_node(name, fn)


### Wire routes, fan-out, and the join barrier

Three ordinary edges leaving `full_review` create parallel work. Passing a
list of all three branch names into the scoring edge creates the barrier:
`score_risk` cannot run until every full-review check has completed.


In [34]:
builder.add_edge(START, "load_case")
builder.add_edge("load_case", "retrieve_capabilities")
builder.add_edge("retrieve_capabilities", "classify_case")
builder.add_conditional_edges("classify_case", route_review,
                              {"missing": "request_documents", "full": "full_review", "quick": "quick_review"})
builder.add_edge("request_documents", END)
builder.add_edge("full_review", "check_freshness")
builder.add_edge("full_review", "screen_sanctions")
builder.add_edge("full_review", "review_audits")
builder.add_edge(["check_freshness", "screen_sanctions", "review_audits"], "score_risk")
builder.add_edge("quick_review", "score_risk")
builder.add_conditional_edges("score_risk", route_after_score,
                              {"remediate": "build_remediation", "draft": "draft_report"})
builder.add_edge("build_remediation", "draft_report")
builder.add_conditional_edges("draft_report", route_after_draft,
                              {"human": "human_approval", "automatic": "automatic_approval"})
builder.add_conditional_edges("human_approval", route_human_decision,
                              {"publish": "publish", "reject": "reject"})
builder.add_edge("automatic_approval", "publish")
builder.add_edge("publish", "verify")
builder.add_edge("verify", END)
builder.add_edge("reject", END)


### Compile with Oracle checkpoints and inspect the native graph

Compiling with `OracleSaver` makes interrupts resumable by thread ID. The
native Mermaid output and node assertion check that the executable graph—not
merely the hand-written diagram above—contains the intended structure.


In [35]:
compiled_graph = builder.compile(checkpointer=checkpointer)
native_mermaid = compiled_graph.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{native_mermaid}\n```"))
expected_nodes = {"load_case", "retrieve_capabilities", "classify_case", "request_documents",
                  "full_review", "check_freshness", "screen_sanctions", "review_audits",
                  "quick_review", "score_risk", "build_remediation", "draft_report",
                  "human_approval", "automatic_approval", "publish", "reject", "verify"}
assert expected_nodes <= set(compiled_graph.get_graph().nodes)
print("Compiled node count:", len(expected_nodes))


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	load_case(load_case)
	retrieve_capabilities(retrieve_capabilities)
	classify_case(classify_case)
	request_documents(request_documents)
	full_review(full_review)
	check_freshness(check_freshness)
	screen_sanctions(screen_sanctions)
	review_audits(review_audits)
	quick_review(quick_review)
	score_risk(score_risk)
	build_remediation(build_remediation)
	draft_report(draft_report)
	human_approval(human_approval)
	automatic_approval(automatic_approval)
	publish(publish)
	reject(reject)
	verify(verify)
	__end__([<p>__end__</p>]):::last
	__start__ --> load_case;
	automatic_approval --> publish;
	build_remediation --> draft_report;
	check_freshness --> score_risk;
	classify_case -. &nbsp;full&nbsp; .-> full_review;
	classify_case -. &nbsp;quick&nbsp; .-> quick_review;
	classify_case -. &nbsp;missing&nbsp; .-> request_documents;
	draft_report -. &nbsp;automatic&nbsp; .-> automatic_approval;
	draft_report -. &nbsp;human&nbsp; .-> human_approval;
	full_review --> check_freshness;
	full_review --> review_audits;
	full_review --> screen_sanctions;
	human_approval -.-> publish;
	human_approval -.-> reject;
	load_case --> retrieve_capabilities;
	publish --> verify;
	quick_review --> score_risk;
	retrieve_capabilities --> classify_case;
	review_audits --> score_risk;
	score_risk -. &nbsp;remediate&nbsp; .-> build_remediation;
	score_risk -. &nbsp;draft&nbsp; .-> draft_report;
	screen_sanctions --> score_risk;
	reject --> __end__;
	request_documents --> __end__;
	verify --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

Compiled node count: 17


## Part V — run the three trajectories end to end

Each invocation has a unique thread ID, so its checkpoint history is
independently recoverable. The high-risk run pauses inside the graph. The
resume command carries a named decision, and LangGraph continues from the
stored interrupt rather than restarting the case. The standard run completes
without a human pause because its score and findings satisfy the explicit
policy. The incomplete run terminates before any risk claim is made.

This is the practical value of branching workflow design: “success” is not
one universal path. A safe evidence request, a short low-risk renewal, and a
fully reviewed high-risk approval are different expected outcomes.


### Run and resume the high-risk trajectory

`initial_state` seeds reducer-backed lists explicitly. Northstar then runs to
the human interrupt. The snapshot proves the next node is durable, and
`Command(resume=...)` supplies a named decision without replaying intake.


In [36]:
def initial_state(supplier_id, thread_id):
    return {
        "thread_id": thread_id,
        "supplier_id": supplier_id,
        "objective": "Review supplier evidence, risk, remediation, and renewal authority.",
        "checks": [], "sandbox_trace": [], "trajectory": [],
    }

high_thread = "visible-high-" + uuid.uuid4().hex[:10]
high_config = {"configurable": {"thread_id": high_thread}}
high_pending = compiled_graph.invoke(initial_state("northstar", high_thread), high_config)
high_snapshot = compiled_graph.get_state(high_config)
assert high_snapshot.next == ("human_approval",)
interrupt_payload = high_pending["__interrupt__"][0].value
pprint({"high_risk_pause": interrupt_payload, "next": high_snapshot.next})

high_completed = compiled_graph.invoke(Command(resume={
    "approved": True,
    "decided_by": "compliance-owner@example.test",
    "comment": "Expired certificate and open audit remediation reviewed.",
}), high_config)
assert high_completed["status"] == "verified"


{'high_risk_pause': {'question': 'Approve this exact report for publication?',
                     'report_digest': 'a7fc72e1377c9d4b7c2820bc4e4c168b1aef6f475b6a91eed48cdbb605216552',
                     'risk_score': 100,
                     'supplier': 'Northstar Textiles'},
 'next': ('human_approval',)}


### Exercise the automatic and missing-evidence routes

Bluebird should finish without an interrupt. Contoso should stop before risk
scoring because required evidence is absent. Assertions make both safe
alternatives part of the executable lesson rather than prose-only claims.


In [37]:
low_thread = "visible-low-" + uuid.uuid4().hex[:10]
low_config = {"configurable": {"thread_id": low_thread}}
low_completed = compiled_graph.invoke(initial_state("bluebird", low_thread), low_config)
assert low_completed["status"] == "verified"
assert "human_approval" not in low_completed["trajectory"]

missing_thread = "visible-missing-" + uuid.uuid4().hex[:10]
missing_config = {"configurable": {"thread_id": missing_thread}}
missing_completed = compiled_graph.invoke(initial_state("contoso", missing_thread), missing_config)
assert missing_completed["status"] == "needs_evidence"
assert "score_risk" not in missing_completed["trajectory"]

pprint({
    "high_risk": {"status": high_completed["status"], "score": high_completed["risk"]["score"],
                  "trajectory": high_completed["trajectory"]},
    "standard": {"status": low_completed["status"], "score": low_completed["risk"]["score"],
                 "trajectory": low_completed["trajectory"]},
    "missing_evidence": {"status": missing_completed["status"],
                         "missing": missing_completed["missing_documents"],
                         "trajectory": missing_completed["trajectory"]},
})


{'high_risk': {'score': 100,
               'status': 'verified',
               'trajectory': ['load_case',
                              'retrieve_capabilities',
                              'classify:full',
                              'full_review_fanout',
                              'check_freshness',
                              'review_audits',
                              'screen_sanctions',
                              'score_risk',
                              'build_remediation',
                              'draft_report',
                              'human_approval',
                              'publish',
                              'verify']},
 'missing_evidence': {'missing': ['restricted_substance_certificate'],
                      'status': 'needs_evidence',
                      'trajectory': ['load_case',
                                     'retrieve_capabilities',
                                     'classify:missing',
                             

### Inspect parallel ordering, sandbox evidence, and checkpoint history

List order among sibling branches may vary, but every branch must appear
before scoring. The selected envelope exposes the tool input, isolation
policy, verified output, and session teardown alongside the Oracle checkpoint
count for the same thread.


In [38]:
# Show the parallel branch evidence and one complete sandbox envelope.
high_branch_positions = {name: high_completed["trajectory"].index(name)
                         for name in ("check_freshness", "screen_sanctions", "review_audits")}
score_position = high_completed["trajectory"].index("score_risk")
assert all(position < score_position for position in high_branch_positions.values())

example = next(item for item in high_completed["sandbox_trace"]
               if item["tool"] == "validate_document_freshness")
print("INPUT")
pprint(example["input"])
print("\nEXECUTION")
pprint(example["execution"])
print("\nOUTPUT")
pprint(example["output"])

checkpoint_history = list(compiled_graph.get_state_history(high_config))
pprint({"parallel_branch_positions": high_branch_positions,
        "score_position": score_position,
        "high_thread_checkpoints": len(checkpoint_history),
        "checkpointer": type(checkpointer).__name__})
assert example["execution"]["session_closed"] is True
assert example["output"]["host_recomputation_matched"] is True


INPUT
{'arguments': {'case': {'annual_spend_usd': 4800000,
                        'documents': {'factory_audit': {'age_days': 120},
                                      'restricted_substance_certificate': {'age_days': 401}},
                        'open_audit_actions': 1,
                        'required_documents': ['restricted_substance_certificate',
                                               'factory_audit'],
                        'risk_class': 'high',
                        'sanctions_matches': 0,
                        'supplier': 'Northstar Textiles',
                        'supplier_id': 'northstar'}},
 'digest': '37d97891682ea23a294c455883f04b74e27220f3f599e685befb164d30866816'}

EXECUTION
{'egress_allowed': False,
 'environment_variable_count': 0,
 'isolation_proven': True,
 'program_digest': '6c760683ba1f5afeaa64c94f49ebfaa102b22e3b4a363b3693f72d72a5eef54e',
 'provider': 'MemoRizz E2BSandboxProvider',
 'session_closed': True,
 'timeout_seconds': 30}

OUTPUT
{'dig

## Part VI — promote a recurring verified trajectory into a Skill

Checkpoints answer where a thread can resume. Workflow recipes answer which
path repeatedly worked. The functions below store a fingerprinted trajectory
with occurrence and success counters. Promotion requires two verified
successes of the same path. It then creates a complete `SKILL.md`, stores its
source recipe ID and SHA, and marks the raw recipe promoted.

This policy is intentionally visible and conservative. A single fluent run,
an unverified publication, or a rejected outcome cannot become procedural
memory. Framework execution alone does not supply this application-specific
learning rule.

```mermaid
flowchart LR
  Run[Completed trajectory] --> Fingerprint[Hash nodes and Tools]
  Fingerprint --> Recipe[(Workflow recipe row)]
  Recipe --> Criteria{At least two verified successes?}
  Criteria -- no --> Candidate[Keep as candidate]
  Criteria -- yes --> Distil[Create parameterised SKILL.md]
  Distil --> Save[(Skillbox body + SHA + lineage)]
  Save --> Future[Future semantic retrieval]
```


### Fingerprint and count verified workflow recipes

The fingerprint covers both node trajectory and executed tool order. Oracle
`MERGE` atomically increments occurrences and verified successes, then the
same transaction reads the current candidate row back for policy evaluation.


In [39]:
def record_workflow(state):
    intent = state["objective"]
    trajectory = state["trajectory"]
    tools = [item["tool"] for item in state["sandbox_trace"]]
    fingerprint = hashlib.sha256(canonical({"trajectory": trajectory, "tools": tools}).encode()).hexdigest()
    recipe_id = fingerprint[:32]
    success = state.get("status") == "verified" and state.get("verification", {}).get("verified")
    vector = embed_text(intent)
    with oracle_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(f'''MERGE INTO {WORKFLOW_TABLE} d
                USING (SELECT :fingerprint fingerprint FROM dual) s ON (d.fingerprint=s.fingerprint)
                WHEN MATCHED THEN UPDATE SET occurrences=d.occurrences+1,
                    successes=d.successes+:success_count,updated_at=SYSTIMESTAMP
                WHEN NOT MATCHED THEN INSERT
                    (recipe_id,fingerprint,intent,trajectory,tools_used,occurrences,successes,promoted,embedding)
                    VALUES (:recipe_id,:fingerprint,:intent,:trajectory,:tools_used,1,:success_count,'N',:embedding)''', {
                "recipe_id": recipe_id, "fingerprint": fingerprint, "intent": intent,
                "trajectory": json.dumps(trajectory), "tools_used": json.dumps(tools),
                "success_count": int(bool(success)), "embedding": array("f", vector),
            })
            cursor.execute(f'''SELECT recipe_id,fingerprint,intent,trajectory,tools_used,
                occurrences,successes,promoted FROM {WORKFLOW_TABLE} WHERE fingerprint=:fingerprint''',
                {"fingerprint": fingerprint})
            raw = cursor.fetchone()
        connection.commit()
    return {"recipe_id": raw[0], "fingerprint": raw[1], "intent": raw[2],
            "trajectory": json_value(raw[3]),
            "tools_used": json_value(raw[4]),
            "occurrences": int(raw[5]), "successes": int(raw[6]), "promoted": raw[7] == "Y"}


### Promote only recurring, uniformly successful recipes

Promotion requires at least two occurrences and no failed occurrence. The new
Skill preserves its source recipe ID, and the recipe row is marked promoted
in Oracle after the full `SKILL.md` is saved.


In [40]:
def promote_if_eligible(recipe):
    if recipe["occurrences"] < 2 or recipe["successes"] != recipe["occurrences"]:
        return {"status": "candidate", "occurrences": recipe["occurrences"]}
    name = "proven-high-risk-supplier-review"
    tools = list(dict.fromkeys(recipe["tools_used"]))
    skill_md = f'''---
name: {name}
description: Reuse the recurring verified high-risk supplier trajectory.
tools: [{', '.join(tools)}]
---
## Steps
1. Retrieve the governed case and relevant procedures.
2. Fan out freshness, sanctions, and audit checks in parallel.
3. Join all check results before scoring risk.
4. Build remediation when policy criteria require it.
5. Pause on the exact report for named approval.
6. Publish and verify the digest before recording success.
'''
    sha = save_skill(name, "Reuse the recurring verified high-risk supplier trajectory.",
                     skill_md, tools, source_workflow_id=recipe["recipe_id"])
    recipe["promoted"] = True
    with oracle_pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                f"UPDATE {WORKFLOW_TABLE} SET promoted='Y',updated_at=SYSTIMESTAMP "
                "WHERE recipe_id=:recipe_id",
                {"recipe_id": recipe["recipe_id"]},
            )
        connection.commit()
    return {"status": "promoted", "skill_name": name, "sha": sha,
            "source_workflow_id": recipe["recipe_id"]}


### Supply the second verified occurrence and inspect lineage

The first Northstar result is recorded, then a fresh Oracle-checkpointed
thread repeats the same approved path. After the second record, the promotion
result is loaded from the Skillbox and its SHA and recipe lineage are asserted.


In [41]:
first_recipe = record_workflow(high_completed)
first_promotion = promote_if_eligible(first_recipe)

repeat_thread = "visible-high-repeat-" + uuid.uuid4().hex[:10]
repeat_config = {"configurable": {"thread_id": repeat_thread}}
repeat_pending = compiled_graph.invoke(initial_state("northstar", repeat_thread), repeat_config)
assert compiled_graph.get_state(repeat_config).next == ("human_approval",)
repeat_completed = compiled_graph.invoke(Command(resume={
    "approved": True, "decided_by": "compliance-owner@example.test",
    "comment": "Second verified execution reviewed.",
}), repeat_config)
second_recipe = record_workflow(repeat_completed)
promotion = promote_if_eligible(second_recipe)
promoted_skill = load_skill(promotion["skill_name"])
pprint({"first_evaluation": first_promotion, "recipe": second_recipe, "promotion": promotion,
        "promoted_skill_sha": promoted_skill["sha"],
        "source_workflow_id": promoted_skill["source_workflow_id"]})
assert promotion["status"] == "promoted" and len(promotion["sha"]) == 64


{'first_evaluation': {'sha': '0602cb38db2abfb9ffd91f938894becf4f78905363687cd4910ece63e39e3661',
                      'skill_name': 'proven-high-risk-supplier-review',
                      'source_workflow_id': 'b714452219e85e1e9e7f0cf25e042614',
                      'status': 'promoted'},
 'promoted_skill_sha': '0602cb38db2abfb9ffd91f938894becf4f78905363687cd4910ece63e39e3661',
 'promotion': {'sha': '0602cb38db2abfb9ffd91f938894becf4f78905363687cd4910ece63e39e3661',
               'skill_name': 'proven-high-risk-supplier-review',
               'source_workflow_id': 'b714452219e85e1e9e7f0cf25e042614',
               'status': 'promoted'},
 'recipe': {'fingerprint': 'b714452219e85e1e9e7f0cf25e042614dd11f95476ed6b31b1c2dd66fb7c751b',
            'intent': 'Review supplier evidence, risk, remediation, and '
                      'renewal authority.',
            'occurrences': 4,
            'promoted': True,
            'recipe_id': 'b714452219e85e1e9e7f0cf25e042614',
            'su

## Key takeaways

- **The graph is not hidden.** Node functions, reducers, parallel edges, join
  barrier, route criteria, interrupt, and resume command are all visible.
- **Expected outcomes are plural.** Missing evidence, standard renewal, named
  high-risk approval, and rejection are legitimate terminal trajectories.
- **The Oracle boundary contains distinct stores.** `OracleSaver` recovers
  working graph state, `OracleDBMemoryStore` holds Agent Memory, and the
  Skillbox holds SHA-versioned procedural documents.
- **Registration is part of the harness.** Python functions are explicitly
  paired with JSON contracts and semantic vectors in the database.
- **Retrieval controls context size.** The model sees a relevant manifest and
  top-k tool contracts rather than every implementation in the system.
- **Parallelism follows independence.** Freshness, sanctions, and audit checks
  run concurrently, then risk waits at an explicit barrier.
- **Tools produce evidence envelopes.** Inputs, execution policy, outputs,
  host recomputation, hashes, and teardown remain inspectable.
- **Policy owns authority.** High-risk results pause for a named person;
  ordinary clear cases can follow a policy decision.
- **Promotion is earned.** Only a recurring, verified trajectory becomes a
  new Skill with SHA and workflow lineage.

## References and further reading

- Anthropic, [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents).
- Anthropic, [Agent Skills overview](https://platform.claude.com/docs/en/agents-and-tools/agent-skills/overview).
- LangGraph, [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api),
  [durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution), and
  [interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts).
- Oracle Agent Memory 26.6, [API index](https://docs.oracle.com/en/database/oracle/agent-memory/26.6/guide/api/index.html)
  and [`OracleAgentMemory`](https://docs.oracle.com/en/database/oracle/agent-memory/26.6/guide/api/agentmemory.html#oracle-agent-memory).
- Oracle, [AI Vector Search User's Guide](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/ai-vector-search-users-guide.pdf).
- E2B, [sandbox documentation](https://e2b.dev/docs).


### Summarise the run and release database resources

The final dictionary gathers the expected outcomes and confirms both durable
Oracle providers. Closing Agent Memory before the shared pool gives the
library a chance to finish its own cleanup while connections are still
available.


In [42]:
final_summary = {
    "high_risk_status": high_completed["status"],
    "standard_status": low_completed["status"],
    "missing_status": missing_completed["status"],
    "parallel_checks": sorted(high_branch_positions),
    "promoted_skill": promotion["skill_name"],
    "promoted_sha": promotion["sha"],
    "checkpoint_provider": type(checkpointer).__name__,
    "memory_substrate": persistence_status["memory_substrate"],
    "toolbox_rows": len(TOOL_REGISTRY),
    "skill_manifest_rows": len(retrieve_skills("supplier compliance", k=5)),
}
pprint(final_summary)
agent_memory.close()
oracle_pool.close(force=True)


{'checkpoint_provider': 'OracleSaver',
 'high_risk_status': 'verified',
 'memory_substrate': 'OracleDBMemoryStore',
 'missing_status': 'needs_evidence',
 'parallel_checks': ['check_freshness', 'review_audits', 'screen_sanctions'],
 'promoted_sha': '0602cb38db2abfb9ffd91f938894becf4f78905363687cd4910ece63e39e3661',
 'promoted_skill': 'proven-high-risk-supplier-review',
 'skill_manifest_rows': 4,
 'standard_status': 'verified',
 'toolbox_rows': 7}
